In [1]:
import os
import numpy as np

from scipy.stats import norm
from scipy.spatial import cKDTree

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel
)

# ============================================================
# FUNCTION 1 - WEEK 10 BAYESIAN OPTIMISATION
# ============================================================
#
# Strategy:
# - Preserve the original objective.
# - Use positive linear Y scaling only.
# - Refit ARD Matern GP including Week 9.
# - Search locally around the best ACTUALLY observed point.
# - Use a tighter empirical trust region than Week 9.
# - NO automatic trust-region expansion this week.
# ============================================================


# ------------------------------------------------------------
# 1. Load Week 10 cumulative data
# ------------------------------------------------------------

if os.path.exists("function1/initial_inputs.npy"):
    # Running from inside week10/
    data_dir = "function1"

elif os.path.exists("../week10/function1/initial_inputs.npy"):
    # Running from inside week9/
    data_dir = "../week10/function1"

else:
    raise FileNotFoundError(
        "Could not find Week 10 Function 1 data."
    )

X = np.load(f"{data_dir}/initial_inputs.npy")
Y = np.load(
    f"{data_dir}/initial_outputs.npy"
).reshape(-1)

best_idx = np.argmax(Y)
best_x = X[best_idx]
best_y = Y[best_idx]

print("================================")
print("DATA")
print("================================")

print("X shape:", X.shape)
print("Y shape:", Y.shape)

print("\nCurrent best:")
print(best_x, "->", best_y)

print("\nY range:")
print("min =", Y.min())
print("max =", Y.max())


# ------------------------------------------------------------
# 2. WEEK 9 CALIBRATION CHECK
# ------------------------------------------------------------
#
# Week 9 selected:
# [0.6259098, 0.73721975]
#
# Week 9 GP used positive linear scaling:
#
# Y_scaled = Y / max(abs(Y))
#
# Prior scaling factor:
# 0.0036060626443634764
#
# Predicted scaled mean:
# 0.00678286
#
# Predicted scaled std:
# 0.01105581
#
# Actual Week 9 output:
# -4.357534823626581e-9
# ------------------------------------------------------------

week9_scale = 0.0036060626443634764

week9_pred_mean_scaled = 0.00678286
week9_pred_std_scaled = 0.01105581

week9_actual_raw = -4.357534823626581e-9
week9_actual_scaled = week9_actual_raw / week9_scale

week9_error_scaled = (
    week9_actual_scaled
    - week9_pred_mean_scaled
)

week9_z_error = (
    week9_error_scaled
    / week9_pred_std_scaled
)

week9_pred_mean_raw = (
    week9_pred_mean_scaled * week9_scale
)

week9_pred_std_raw = (
    week9_pred_std_scaled * week9_scale
)

print("\n================================")
print("WEEK 9 CALIBRATION CHECK")
print("================================")

print("Predicted raw mean:")
print(week9_pred_mean_raw)

print("\nPredicted raw std:")
print(week9_pred_std_raw)

print("\nActual:")
print(week9_actual_raw)

print("\nPrediction error / predicted std:")
print(week9_z_error)


# ------------------------------------------------------------
# 3. Positive linear Y scaling
# ------------------------------------------------------------
#
# IMPORTANT:
#
# This does NOT change the optimisation objective.
#
# Multiplication/division by a positive constant preserves:
# - ordering
# - argmax
# - sign
#
# No log transform and no absolute-value objective transform.
# ------------------------------------------------------------

y_scale = np.max(np.abs(Y))

Y_scaled = Y / y_scale

best_y_scaled = np.max(Y_scaled)

print("\n================================")
print("WEEK 10 Y SCALING")
print("================================")

print("Scale factor:")
print(y_scale)

print("\nScaled Y range:")
print(Y_scaled.min(), "to", Y_scaled.max())

print("\nBest scaled Y:")
print(best_y_scaled)


# ------------------------------------------------------------
# 4. Fit ARD Matern GP
# ------------------------------------------------------------

kernel = (
    ConstantKernel(
        1.0,
        constant_value_bounds=(1e-3, 1e3)
    )
    *
    Matern(
        length_scale=np.ones(2) * 0.2,
        length_scale_bounds=(0.005, 2.0),
        nu=2.5
    )
    +
    WhiteKernel(
        noise_level=1e-6,
        noise_level_bounds=(1e-8, 1e-1)
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=False,
    n_restarts_optimizer=30,
    random_state=42
)

gp.fit(X, Y_scaled)


print("\n================================")
print("GP FIT")
print("================================")

print("\nFitted kernel:")
print(gp.kernel_)

lengthscales = gp.kernel_.k1.k2.length_scale

inverse_ls = 1.0 / lengthscales
relative_sensitivity = (
    inverse_ls / inverse_ls.sum()
)

print("\nARD lengthscales:")
print(lengthscales)

print(
    "\nNormalised inverse-lengthscale sensitivity:"
)
print(relative_sensitivity)


# ------------------------------------------------------------
# 5. Expected Improvement
# ------------------------------------------------------------

def expected_improvement(
    mu,
    sigma,
    best,
    xi=0.0
):

    improvement = (
        mu - best - xi
    )

    valid = sigma > 1e-12

    Z = np.zeros_like(mu)

    Z[valid] = (
        improvement[valid]
        / sigma[valid]
    )

    EI = np.zeros_like(mu)

    EI[valid] = (
        improvement[valid]
        * norm.cdf(Z[valid])
        +
        sigma[valid]
        * norm.pdf(Z[valid])
    )

    return EI


# ------------------------------------------------------------
# 6. Empirical distance around incumbent
# ------------------------------------------------------------
#
# Determine how close the incumbent already is to another
# observed point.
#
# Week 9 used roughly 2 * nearest-neighbour distance as the
# empirical cap.
#
# Week 10 uses only 1 * nearest-neighbour distance because
# last week's attempted movement away from the incumbent
# performed poorly.
# ------------------------------------------------------------

other_mask = np.arange(len(X)) != best_idx

distances_to_best = np.linalg.norm(
    X[other_mask] - best_x,
    axis=1
)

nearest_distance = (
    distances_to_best.min()
)

empirical_cap = min(
    nearest_distance,
    0.05
)

print("\n================================")
print("EMPIRICAL LOCAL SCALE")
print("================================")

print("Nearest observed point to best:")
print(nearest_distance)

print("\nEmpirical cap:")
print(empirical_cap)


# ------------------------------------------------------------
# 7. Tight ARD trust region
# ------------------------------------------------------------
#
# Width depends on:
# - fitted ARD lengthscales
# - actual sampling density near incumbent
#
# x dimensions estimated to vary rapidly receive narrower
# search widths.
#
# No boundary-triggered expansion this week.
# ------------------------------------------------------------

trust_half_width = np.clip(
    0.25 * lengthscales,
    0.005,
    empirical_cap
)

lower = np.maximum(
    0.0,
    best_x - trust_half_width
)

upper = np.minimum(
    1.0,
    best_x + trust_half_width
)

print("\n================================")
print("WEEK 10 TRUST REGION")
print("================================")

print("Centre:")
print(best_x)

print("\nHalf-widths:")
print(trust_half_width)

print("\nLower bounds:")
print(lower)

print("\nUpper bounds:")
print(upper)


# ------------------------------------------------------------
# 8. Dense 2D trust-region grid
# ------------------------------------------------------------

n_grid = 601

x1_grid = np.linspace(
    lower[0],
    upper[0],
    n_grid
)

x2_grid = np.linspace(
    lower[1],
    upper[1],
    n_grid
)

xx1, xx2 = np.meshgrid(
    x1_grid,
    x2_grid
)

candidates = np.column_stack([
    xx1.ravel(),
    xx2.ravel()
])


# ------------------------------------------------------------
# 9. Remove near-duplicates
# ------------------------------------------------------------

tree = cKDTree(X)

distance, _ = tree.query(
    candidates,
    k=1
)

candidates = candidates[
    distance > 0.01
]

print("\nCandidates after duplicate filtering:")
print(len(candidates))


# ------------------------------------------------------------
# 10. GP predictions
# ------------------------------------------------------------

mu, sigma = gp.predict(
    candidates,
    return_std=True
)


# ------------------------------------------------------------
# 11. PRIMARY EI
# ------------------------------------------------------------

EI = expected_improvement(
    mu,
    sigma,
    best_y_scaled,
    xi=0.0
)

ei_idx = np.argmax(EI)

print("\n================================")
print("PRIMARY EI - TRUST REGION")
print("================================")

print("candidate =", candidates[ei_idx])

print("scaled mean =", mu[ei_idx])
print("scaled std  =", sigma[ei_idx])

print(
    "raw mean =",
    mu[ei_idx] * y_scale
)

print(
    "raw std =",
    sigma[ei_idx] * y_scale
)

print("EI =", EI[ei_idx])


# ------------------------------------------------------------
# 12. EI sensitivity
# ------------------------------------------------------------

print("\n================================")
print("EI SENSITIVITY")
print("================================\n")

for xi in [0.0, 0.001, 0.005, 0.01]:

    EI_test = expected_improvement(
        mu,
        sigma,
        best_y_scaled,
        xi=xi
    )

    idx = np.argmax(EI_test)

    print(
        "xi =", xi,
        "\n candidate =", candidates[idx],
        "\n scaled mean =", mu[idx],
        "\n scaled std =", sigma[idx],
        "\n EI =", EI_test[idx],
        "\n"
    )


# ------------------------------------------------------------
# 13. Highest predicted mean
# ------------------------------------------------------------

mean_idx = np.argmax(mu)

print("\n================================")
print("HIGHEST PREDICTED MEAN")
print("================================")

print("candidate =", candidates[mean_idx])

print("scaled mean =", mu[mean_idx])
print("scaled std  =", sigma[mean_idx])

print(
    "raw mean =",
    mu[mean_idx] * y_scale
)

print(
    "raw std =",
    sigma[mean_idx] * y_scale
)


# ------------------------------------------------------------
# 14. UCB diagnostics
# ------------------------------------------------------------

print("\n================================")
print("UCB DIAGNOSTICS")
print("================================\n")

for beta in [
    0.05,
    0.1,
    0.25,
    0.5,
    1.0
]:

    UCB = (
        mu
        + beta * sigma
    )

    idx = np.argmax(UCB)

    print(
        f"beta={beta}",
        "\n candidate =", candidates[idx],
        "\n scaled mean =", mu[idx],
        "\n scaled std =", sigma[idx],
        "\n raw mean =", mu[idx] * y_scale,
        "\n raw std =", sigma[idx] * y_scale,
        "\n UCB =", UCB[idx],
        "\n"
    )


# ------------------------------------------------------------
# 15. Boundary diagnostic
# ------------------------------------------------------------
#
# IMPORTANT:
# If the selected acquisitions hit a trust-region boundary,
# we will PRINT that fact but NOT expand automatically.
# ------------------------------------------------------------

def boundary_status(x, lower, upper, tol=1e-6):

    status = []

    for j in range(len(x)):

        if abs(x[j] - lower[j]) <= tol:
            status.append(
                f"x{j+1}=LOWER"
            )

        elif abs(x[j] - upper[j]) <= tol:
            status.append(
                f"x{j+1}=UPPER"
            )

    if not status:
        return "interior"

    return ", ".join(status)


print("\n================================")
print("BOUNDARY CHECK")
print("================================")

print(
    "EI:",
    boundary_status(
        candidates[ei_idx],
        lower,
        upper
    )
)

print(
    "Highest mean:",
    boundary_status(
        candidates[mean_idx],
        lower,
        upper
    )
)

for beta in [
    0.05,
    0.1,
    0.25,
    0.5,
    1.0
]:

    UCB = mu + beta * sigma
    idx = np.argmax(UCB)

    print(
        f"UCB beta={beta}:",
        boundary_status(
            candidates[idx],
            lower,
            upper
        )
    )

DATA
X shape: (19, 2)
Y shape: (19,)

Current best:
[0.73102363 0.73299988] -> 7.710875114502849e-16

Y range:
min = -0.0036060626443634764
max = 7.710875114502849e-16

WEEK 9 CALIBRATION CHECK
Predicted raw mean:
2.445941806794725e-05

Predicted raw std:
3.9867943444180163e-05

Actual:
-4.357534823626581e-09

Prediction error / predicted std:
-0.6136202043384319

WEEK 10 Y SCALING
Scale factor:
0.0036060626443634764

Scaled Y range:
-1.0 to 2.1383086970370515e-13

Best scaled Y:
2.1383086970370515e-13


/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-08. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(



GP FIT

Fitted kernel:
0.254**2 * Matern(length_scale=[2, 0.0359], nu=2.5) + WhiteKernel(noise_level=1e-08)

ARD lengthscales:
[2.         0.03585027]

Normalised inverse-lengthscale sensitivity:
[0.01760948 0.98239052]

EMPIRICAL LOCAL SCALE
Nearest observed point to best:
0.026278458561148094

Empirical cap:
0.026278458561148094

WEEK 10 TRUST REGION
Centre:
[0.73102363 0.73299988]

Half-widths:
[0.02627846 0.00896257]

Lower bounds:
[0.70474517 0.72403731]

Upper bounds:
[0.75730209 0.74196244]

Candidates after duplicate filtering:
245776

PRIMARY EI - TRUST REGION
candidate = [0.75730209 0.7358679 ]
scaled mean = 0.002684561286723064
scaled std  = 0.004035082840813471
raw mean = 9.68069617255639e-06
raw std = 1.4550761499169512e-05
EI = 0.0032957299558287394

EI SENSITIVITY

xi = 0.0 
 candidate = [0.75730209 0.7358679 ] 
 scaled mean = 0.002684561286723064 
 scaled std = 0.004035082840813471 
 EI = 0.0032957299558287394 

xi = 0.001 
 candidate = [0.75730209 0.73589777] 
 scaled

In [2]:
# ============================================================
# FINAL FUNCTION 1 - WEEK 10 SELECTION
# ============================================================
#
# The Week 9 standardised residual was only -0.61 sigma,
# but the absolute GP prediction remained badly mis-scaled
# relative to the tiny objective values.
#
# Therefore the GP is used primarily for LOCAL RANKING,
# not as a literal predictor of function magnitude.
#
# EI, posterior mean and every tested UCB beta all push x1
# to the upper edge of the deliberately tight trust region.
#
# We DO NOT expand again because the Week 9 boundary expansion
# produced a poor realised result.
#
# Since uncertainty calibration is questionable for F1,
# select the highest posterior mean inside the controlled
# trust region rather than rewarding additional uncertainty.

final_idx = np.argmax(mu)

week10_candidate = candidates[final_idx]

print("Week 10 Function 1 candidate:")
print(week10_candidate)

print("\nScaled predicted mean:")
print(mu[final_idx])

print("\nScaled predicted std:")
print(sigma[final_idx])

print("\nRaw predicted mean:")
print(mu[final_idx] * y_scale)

print("\nRaw predicted std:")
print(sigma[final_idx] * y_scale)

portal = "-".join(
    f"{x:.6f}"
    for x in week10_candidate
)

print("\nPortal format:")
print(portal)

Week 10 Function 1 candidate:
[0.75730209 0.73556915]

Scaled predicted mean:
0.002703728252790305

Scaled predicted std:
0.003928864834318074

Raw predicted mean:
9.74981345289725e-06

Raw predicted std:
1.4167732713787705e-05

Portal format:
0.757302-0.735569
